# Simple timing sketch

It is not a full benchmark suite, but it is handy for quick local comparisons.

In [1]:
import os, sys, platform; print(f"{platform.python_implementation()} {platform.python_version()} | {platform.system()} {platform.release()} | CPU: {platform.processor() or platform.machine()} | logical CPUs: {os.cpu_count()}")

CPython 3.12.13 | Windows 11 | CPU: Intel64 Family 6 Model 151 Stepping 2, GenuineIntel | logical CPUs: 24


### Reading uncompressed CT image

In [2]:
# pip install "dicomsdl[numpy]"
import dicomsdl as dicom
import numpy as np

def func_dicomsdl_unc():
    df = dicom.read_file("CT1_UNC")
    return df.to_array()
    
%timeit func_dicomsdl_unc()


285 μs ± 18.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [3]:
def func_dicomsdl_unc_fastest():
    df = dicom.read_file("CT1_UNC")
    return df.to_array_view()

%timeit func_dicomsdl_unc_fastest()


53.1 μs ± 4.63 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [4]:
# pip install pydicom
from pydicom import dcmread

def func_pydicom_unc():
    ds = dcmread("CT1_UNC")
    return ds.pixel_array

%timeit func_pydicom_unc()


1.26 ms ± 33.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
def func_pydicom_unc_fastest():
    ds = dcmread("CT1_UNC")
    return np.frombuffer(
        ds.PixelData,
        dtype=np.int16,
    ).reshape(512, 512)

%timeit func_pydicom_unc_fastest()


983 μs ± 34.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [6]:
# pip install python-gdcm
import ctypes
import gdcm

def func_gdcm_unc():
    reader = gdcm.ImageReader()
    reader.SetFileName("CT1_UNC")
    reader.Read()
    return reader.GetImage().GetBuffer()

%timeit func_gdcm_unc()

2.55 ms ± 293 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [7]:
def func_gdcm_unc_fastest():
    reader = gdcm.ImageReader()
    reader.SetFileName("CT1_UNC")
    if not reader.Read():
        raise RuntimeError("GDCM failed to read CT1_UNC")

    byte_value = reader.GetImage().GetDataElement().GetByteValue()
    ptr = int(byte_value.GetVoidPointer())
    length = int(str(byte_value.GetLength()))
    return np.frombuffer(
        ctypes.string_at(ptr, length),
        dtype=np.int16,
    ).reshape(512, 512)

%timeit func_gdcm_unc_fastest()


449 μs ± 11.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### Reading jpeg-ls compressed CT image

In [8]:
def func_dicomsdl_jlsn():
    df = dicom.read_file("CT2_JLSN")
    return df.to_array()
    
%timeit func_dicomsdl_jlsn()


3 ms ± 208 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
# pip install pyjpegls
def func_pydicom_jlsn():
    ds = dcmread("CT2_JLSN")
    ds.pixel_array_options(decoding_plugin="pyjpegls")
    return ds.pixel_array
    
%timeit func_pydicom_jlsn()


4.24 ms ± 189 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [10]:
# pip install python-gdcm
def func_pydicom_gdcm_jlsn():
    ds = dcmread("CT2_JLSN")
    ds.pixel_array_options(decoding_plugin="gdcm")
    return ds.pixel_array
    
%timeit func_pydicom_gdcm_jlsn()


6.88 ms ± 380 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
